# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', None))
print("License:", getattr(metadata, 'license', None))

## 2. Data Overview
Review available record sets and fields. All references use the entity `@id` for maximum reliability and traceability.

In [ ]:
# Retrieve and display all record sets and referenced fields by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")

for rs in record_sets:
    print(f"- Record Set @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List field IDs
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields (@id):")
        for field in rs.fields:
            print(f"    - {field.id}")
    print()

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All entity accesses are by `@id`. You may adjust the record set for other subsets as needed.

In [ ]:
dataframes = dict()

# List all record set @id values
record_set_ids = [rs.id for rs in record_sets]
print("Available Record Set @id values:")
print(record_set_ids)

# Pick the main record set, typically the first for this dataset
main_record_set_id = record_set_ids[0]
print(f"\nLoading records from main record set: {main_record_set_id}")
main_rs = next(rs for rs in record_sets if rs.id == main_record_set_id)

records = list(dataset.records(record_set=main_record_set_id))
main_df = pd.DataFrame(records)
dataframes[main_record_set_id] = main_df
print(f"Loaded {len(main_df)} records with columns:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping. We will choose fields by their `@id`, accessible from the record set's field list above.

In [ ]:
# Example: Select a numeric field for mental health measurement for filtering and normalization

# List all fields of the main record set
main_fields = main_rs.fields
for field in main_fields:
    print(f"Field @id: {field.id} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")

# For illustration, suppose one field is GAD-7 Score with @id 'gad7_score'
# In Croissant, fields are often named accordingly; adapt these IDs for your dataset.
gad7_field_id = None
education_field_id = None
for field in main_fields:
    if 'gad' in field.id.lower():
        gad7_field_id = field.id
    if 'education' in field.id.lower():
        education_field_id = field.id

if gad7_field_id is None:
    print("Could not automatically find GAD-7 field; please check the field IDs above.")
else:
    print(f"Using GAD-7 field @id: {gad7_field_id}")

# Use a threshold for demonstration, e.g., respondents with severe anxiety
if gad7_field_id in main_df.columns:
    try:
        main_df[gad7_field_id] = pd.to_numeric(main_df[gad7_field_id], errors='coerce')
    except:
        pass
    threshold = 10
    filtered_df = main_df[main_df[gad7_field_id] > threshold].copy()
    print(f"Filtered records with {gad7_field_id} > {threshold}: {len(filtered_df)} cases")
    print(filtered_df.head())

    # Normalize the GAD-7 field
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by education field if available
    if education_field_id and education_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(education_field_id)[gad7_field_id].mean().to_frame()
        print(f"\nGrouped mean {gad7_field_id} by {education_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {gad7_field_id} is not present in columns. Please check dataset fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using standard Python tools like matplotlib and seaborn. We show a histogram of GAD-7 scores and, if available, boxplots by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
if gad7_field_id and gad7_field_id in main_df.columns:
    sns.histplot(main_df[gad7_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {gad7_field_id}")
    plt.xlabel(gad7_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print(f"No numeric field {gad7_field_id} found for visualization.")

# Boxplot by education level
if gad7_field_id and education_field_id and (
    gad7_field_id in main_df.columns and education_field_id in main_df.columns
):
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=education_field_id, y=gad7_field_id, data=main_df)
    plt.title(f"{gad7_field_id} Scores by {education_field_id}")
    plt.xlabel(education_field_id)
    plt.ylabel(gad7_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
We loaded the FAIR² mental health survey dataset, reviewed its structure, extracted the records using their `@id`, and performed basic exploratory analysis including filtering by GAD-7 score, normalization, grouping by education, and visualization.

Change field selections or grouping fields using the `@id` values from Section 2 according to your analysis needs.